# GeoTrade AI — Stage 2: XGBoost ML Model Training
Trains Signal Classifier (BUY/SELL/HOLD) + Drift/Volatility Regressors.
Run cells top to bottom. Final cell saves `.pkl` files to `backend/app/ml/saved_models/`.

In [ ]:
# ── Cell 1: Install dependencies ──────────────────────────────────────────
!pip install yfinance xgboost scikit-learn pandas numpy joblib matplotlib seaborn --quiet

In [ ]:
# ── Cell 2: Imports & Config ──────────────────────────────────────────────
import warnings, os, math
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import yfinance as yf
import joblib
import matplotlib.pyplot as plt
import seaborn as sns

from xgboost import XGBClassifier, XGBRegressor
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    classification_report, confusion_matrix,
    mean_absolute_error, r2_score
)

# ── Configuration ─────────────────────────────────────────────────────────
ASSETS = {
    'GOLD':      'GC=F',
    'OIL_BRENT': 'BZ=F',
    'SP500':     '^GSPC',
    'BTCUSD':    'BTC-USD',
}
PERIOD        = '5y'        # 5 years of daily data
SIGNAL_THRESH = 0.02        # ±2% 5-day return threshold for BUY/SELL
HORIZON       = 5           # predict 5-day forward return
SAVE_DIR      = '../backend/app/ml/saved_models'
os.makedirs(SAVE_DIR, exist_ok=True)
print('Config OK. Save dir:', os.path.abspath(SAVE_DIR))

In [ ]:
# ── Cell 3: Download Price Data from yfinance ─────────────────────────────
raw_data = {}
for name, ticker in ASSETS.items():
    df = yf.download(ticker, period=PERIOD, interval='1d', progress=False)
    df.columns = [c[0].lower() if isinstance(c, tuple) else c.lower() for c in df.columns]
    df.dropna(inplace=True)
    raw_data[name] = df
    print(f'{name:12s} → {len(df):4d} rows  ({df.index[0].date()} to {df.index[-1].date()})')

# Also download VIX as a Fear/Tension proxy
vix = yf.download('^VIX', period=PERIOD, interval='1d', progress=False)
vix.columns = [c[0].lower() if isinstance(c, tuple) else c.lower() for c in vix.columns]
vix = vix[['close']].rename(columns={'close': 'vix'})
print(f'VIX         → {len(vix):4d} rows')

In [ ]:
# ── Cell 4: Technical Indicator Functions ─────────────────────────────────
def compute_rsi(series, period=14):
    delta = series.diff()
    gain  = delta.clip(lower=0).rolling(period).mean()
    loss  = (-delta.clip(upper=0)).rolling(period).mean()
    rs    = gain / loss.replace(0, np.nan)
    return 100 - (100 / (1 + rs))

def compute_macd(series, fast=12, slow=26, signal=9):
    ema_fast   = series.ewm(span=fast, adjust=False).mean()
    ema_slow   = series.ewm(span=slow, adjust=False).mean()
    macd_line  = ema_fast - ema_slow
    signal_line= macd_line.ewm(span=signal, adjust=False).mean()
    return macd_line - signal_line  # MACD histogram

def compute_bollinger_pct(series, period=20):
    ma  = series.rolling(period).mean()
    std = series.rolling(period).std()
    upper = ma + 2 * std
    lower = ma - 2 * std
    return (series - lower) / (upper - lower).replace(0, np.nan)

def compute_atr(df, period=14):
    tr = pd.concat([
        df['high'] - df['low'],
        (df['high'] - df['close'].shift()).abs(),
        (df['low']  - df['close'].shift()).abs(),
    ], axis=1).max(axis=1)
    return tr.rolling(period).mean()

print('Indicator functions ready.')

In [ ]:
# ── Cell 5: Feature Engineering ──────────────────────────────────────────
def build_features(df, vix_df, asset_name):
    f = pd.DataFrame(index=df.index)

    # Price features
    f['close']        = df['close']
    f['return_1d']    = df['close'].pct_change(1)
    f['return_5d']    = df['close'].pct_change(5)
    f['return_20d']   = df['close'].pct_change(20)

    # Volatility
    f['vol_20d']      = df['close'].pct_change().rolling(20).std() * math.sqrt(252)
    f['atr_14']       = compute_atr(df, 14) / df['close']  # normalised

    # Momentum
    f['rsi_14']       = compute_rsi(df['close'], 14)
    f['macd_hist']    = compute_macd(df['close'])
    f['bollinger_pct']= compute_bollinger_pct(df['close'], 20)

    # Volume
    if 'volume' in df.columns:
        f['vol_ratio'] = df['volume'] / df['volume'].rolling(20).mean()
    else:
        f['vol_ratio'] = 1.0

    # Geopolitical Fear Proxy (VIX)
    f = f.join(vix_df, how='left')
    f['vix'] = f['vix'].fillna(method='ffill')
    f['vix_change'] = f['vix'].pct_change(5)

    # Major Geopolitical Event flags (synthetic GTI proxy)
    crisis_dates = [
        ('2020-03-01', '2020-04-30', 9.0),   # COVID crash
        ('2022-02-24', '2022-05-01', 8.5),   # Russia-Ukraine
        ('2022-06-01', '2022-09-30', 6.0),   # Energy crisis
        ('2023-10-07', '2024-01-01', 7.5),   # Gaza conflict
        ('2024-04-01', '2024-06-30', 5.5),   # Middle East escalation
    ]
    f['gti_proxy'] = 35.0  # baseline low tension
    for start, end, score in crisis_dates:
        mask = (f.index >= start) & (f.index <= end)
        f.loc[mask, 'gti_proxy'] = score * 10  # scale to 0-100

    # Target: 5-day forward return
    f['fwd_return_5d'] = df['close'].pct_change(HORIZON).shift(-HORIZON)

    # Label: BUY / SELL / HOLD
    f['label'] = 'HOLD'
    f.loc[f['fwd_return_5d'] >  SIGNAL_THRESH, 'label'] = 'BUY'
    f.loc[f['fwd_return_5d'] < -SIGNAL_THRESH, 'label'] = 'SELL'

    f.dropna(inplace=True)
    print(f'{asset_name}: {len(f)} rows | BUY={sum(f.label=="BUY")} SELL={sum(f.label=="SELL")} HOLD={sum(f.label=="HOLD")}')
    return f

datasets = {}
for name, df in raw_data.items():
    datasets[name] = build_features(df, vix, name)

In [ ]:
# ── Cell 6: Visualise Label Distribution ─────────────────────────────────
fig, axes = plt.subplots(1, len(datasets), figsize=(16, 4))
colors = {'BUY': '#22c55e', 'SELL': '#ef4444', 'HOLD': '#64748b'}
for ax, (name, df) in zip(axes, datasets.items()):
    counts = df['label'].value_counts()
    ax.bar(counts.index, counts.values, color=[colors[l] for l in counts.index])
    ax.set_title(name)
    ax.set_xlabel('Signal')
    ax.set_ylabel('Count')
plt.suptitle('Label Distribution per Asset', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ── Cell 7: Feature Columns ───────────────────────────────────────────────
FEATURE_COLS = [
    'return_1d', 'return_5d', 'return_20d',
    'vol_20d', 'atr_14',
    'rsi_14', 'macd_hist', 'bollinger_pct',
    'vol_ratio', 'vix', 'vix_change', 'gti_proxy'
]
print('Feature set:', FEATURE_COLS)

In [ ]:
# ── Cell 8: Train/Test Split (time-series aware — NO shuffling) ───────────
ASSET_TO_TRAIN = 'GOLD'   # Change this to train a different asset
df = datasets[ASSET_TO_TRAIN].copy()

split = int(len(df) * 0.80)
train_df = df.iloc[:split]
test_df  = df.iloc[split:]

X_train = train_df[FEATURE_COLS]
X_test  = test_df[FEATURE_COLS]

# Classification labels
le = LabelEncoder()
y_train_cls = le.fit_transform(train_df['label'])
y_test_cls  = le.transform(test_df['label'])

# Regression targets
y_train_drift = train_df['fwd_return_5d'] * (252/5)   # annualise
y_test_drift  = test_df['fwd_return_5d']  * (252/5)
y_train_vol   = train_df['vol_20d']
y_test_vol    = test_df['vol_20d']

print(f'Train: {len(train_df)} rows | Test: {len(test_df)} rows')
print(f'Classes: {le.classes_}')

In [ ]:
# ── Cell 9: Train Signal Classifier (XGBoost) ─────────────────────────────
clf = XGBClassifier(
    n_estimators=400,
    max_depth=5,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    use_label_encoder=False,
    eval_metric='mlogloss',
    random_state=42,
    n_jobs=-1,
)
clf.fit(
    X_train, y_train_cls,
    eval_set=[(X_test, y_test_cls)],
    verbose=50,
)
print('\nSignal Classifier training complete!')

In [ ]:
# ── Cell 10: Evaluate Signal Classifier ──────────────────────────────────
y_pred_cls = clf.predict(X_test)
print('=== Classification Report ===')
print(classification_report(y_test_cls, y_pred_cls, target_names=le.classes_))

# Confusion Matrix
cm = confusion_matrix(y_test_cls, y_pred_cls)
plt.figure(figsize=(7, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=le.classes_, yticklabels=le.classes_)
plt.title(f'{ASSET_TO_TRAIN} — Signal Classifier Confusion Matrix')
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.tight_layout()
plt.show()

# Feature importance
importances = pd.Series(clf.feature_importances_, index=FEATURE_COLS).sort_values(ascending=False)
print('\nTop Features:')
print(importances.head(8).to_string())

In [ ]:
# ── Cell 11: Train Drift Regressor ────────────────────────────────────────
drift_reg = XGBRegressor(
    n_estimators=300, max_depth=4, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8, random_state=42, n_jobs=-1
)
drift_reg.fit(X_train, y_train_drift, eval_set=[(X_test, y_test_drift)], verbose=50)

drift_preds = drift_reg.predict(X_test)
print(f'\nDrift Regressor — MAE: {mean_absolute_error(y_test_drift, drift_preds):.4f}')
print(f'Drift Regressor — R²:  {r2_score(y_test_drift, drift_preds):.4f}')

# ── Train Volatility Regressor ────────────────────────────────────────────
vol_reg = XGBRegressor(
    n_estimators=300, max_depth=4, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8, random_state=42, n_jobs=-1
)
vol_reg.fit(X_train, y_train_vol, eval_set=[(X_test, y_test_vol)], verbose=50)

vol_preds = vol_reg.predict(X_test)
print(f'\nVolatility Regressor — MAE: {mean_absolute_error(y_test_vol, vol_preds):.4f}')
print(f'Volatility Regressor — R²:  {r2_score(y_test_vol, vol_preds):.4f}')

In [ ]:
# ── Cell 12: Save .pkl Files ──────────────────────────────────────────────
symbol = ASSET_TO_TRAIN
joblib.dump(clf,       f'{SAVE_DIR}/{symbol}_signal_classifier.pkl')
joblib.dump(drift_reg, f'{SAVE_DIR}/{symbol}_drift_regressor.pkl')
joblib.dump(vol_reg,   f'{SAVE_DIR}/{symbol}_vol_regressor.pkl')
joblib.dump(le,        f'{SAVE_DIR}/{symbol}_label_encoder.pkl')
joblib.dump(FEATURE_COLS, f'{SAVE_DIR}/{symbol}_feature_cols.pkl')

print(f'Saved 5 files for {symbol}:')
for f in os.listdir(SAVE_DIR):
    if symbol in f:
        size = os.path.getsize(f'{SAVE_DIR}/{f}') / 1024
        print(f'  {f}  ({size:.1f} KB)')

In [ ]:
# ── Cell 13: Quick Inference Test (simulate live prediction) ──────────────
# Load saved models and run a prediction on the last row of test data
clf_loaded    = joblib.load(f'{SAVE_DIR}/{symbol}_signal_classifier.pkl')
drift_loaded  = joblib.load(f'{SAVE_DIR}/{symbol}_drift_regressor.pkl')
vol_loaded    = joblib.load(f'{SAVE_DIR}/{symbol}_vol_regressor.pkl')
le_loaded     = joblib.load(f'{SAVE_DIR}/{symbol}_label_encoder.pkl')

live_row = X_test.iloc[[-1]]  # last test row as 'live' data

proba     = clf_loaded.predict_proba(live_row)[0]
signal_id = proba.argmax()
signal    = le_loaded.classes_[signal_id]
confidence= proba[signal_id]
drift_val = float(drift_loaded.predict(live_row)[0])
vol_val   = float(vol_loaded.predict(live_row)[0])

print(f'=== Live Inference Test for {symbol} ===')
print(f'Signal     : {signal}')
print(f'Confidence : {confidence:.2%}')
print(f'Pred Drift : {drift_val:.4f} ({drift_val*100:.2f}% annualised)')
print(f'Pred Vol   : {vol_val:.4f} ({vol_val*100:.2f}% annualised)')
print(f'Proba      : BUY={proba[list(le_loaded.classes_).index("BUY")]:.2%}  SELL={proba[list(le_loaded.classes_).index("SELL")]:.2%}  HOLD={proba[list(le_loaded.classes_).index("HOLD")]:.2%}')
print('\nAll done! .pkl files are ready for the backend.')

In [ ]:
# ── Cell 14: Train ALL Assets (run after tuning on GOLD) ─────────────────
# Uncomment and run this cell to train models for all 4 assets at once

# for asset_name in ASSETS.keys():
#     print(f'\n{'='*50}')
#     print(f'Training {asset_name}...')
#     df = datasets[asset_name].copy()
#     split = int(len(df) * 0.80)
#     X_tr, X_te = df.iloc[:split][FEATURE_COLS], df.iloc[split:][FEATURE_COLS]
#     le_a = LabelEncoder()
#     y_tr_c = le_a.fit_transform(df.iloc[:split]['label'])
#     y_te_c = le_a.transform(df.iloc[split:]['label'])
#     clf_a = XGBClassifier(n_estimators=400, max_depth=5, learning_rate=0.05, subsample=0.8, colsample_bytree=0.8, use_label_encoder=False, eval_metric='mlogloss', random_state=42, n_jobs=-1)
#     clf_a.fit(X_tr, y_tr_c, verbose=False)
#     drift_a = XGBRegressor(n_estimators=300, max_depth=4, learning_rate=0.05, random_state=42, n_jobs=-1)
#     drift_a.fit(X_tr, df.iloc[:split]['fwd_return_5d'] * (252/5), verbose=False)
#     vol_a = XGBRegressor(n_estimators=300, max_depth=4, learning_rate=0.05, random_state=42, n_jobs=-1)
#     vol_a.fit(X_tr, df.iloc[:split]['vol_20d'], verbose=False)
#     joblib.dump(clf_a,   f'{SAVE_DIR}/{asset_name}_signal_classifier.pkl')
#     joblib.dump(drift_a, f'{SAVE_DIR}/{asset_name}_drift_regressor.pkl')
#     joblib.dump(vol_a,   f'{SAVE_DIR}/{asset_name}_vol_regressor.pkl')
#     joblib.dump(le_a,    f'{SAVE_DIR}/{asset_name}_label_encoder.pkl')
#     joblib.dump(FEATURE_COLS, f'{SAVE_DIR}/{asset_name}_feature_cols.pkl')
#     preds = clf_a.predict(X_te)
#     from sklearn.metrics import accuracy_score
#     print(f'{asset_name} Accuracy: {accuracy_score(y_te_c, preds):.2%}')
#     print(f'Saved to {SAVE_DIR}/')
print('Uncomment the block above to train all assets.')